## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Pemusatan & Deteksi Outlier](images/img_02_central_tendency_outliers.png)

```
                      +-------------------+
                      |   ANATOMI BOXPLOT |
                      +-------------------+
                      
   Outlier Ringan (< Q1 - 1.5*IQR)       Outlier Atas (> Q3 + 1.5*IQR)
        *   *                                       *       *
   -----+---+--------[=======|=======]--------------+-------+----
        |            |       |       |              |
    Lower Whisker   Q1    Median    Q3         Upper Whisker
   (Q1 - 1.5*IQR)  (25%)   (50%)   (75%)       (Q3 + 1.5*IQR)
                     |<----- IQR ---->|
```


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (11, 5)

# Memuat dataset
df = pd.read_csv("../datasets/01_ecommerce_sales_eda.csv")
print("Data berhasil dimuat. Total baris:", len(df))


## 🧮 3. Perhitungan Parameter Pemusatan dan Penyebaran Data


In [ ]:
target_col = 'total_amount_k'

mean_val = df[target_col].mean()
median_val = df[target_col].median()
mode_val = df[target_col].mode()[0]
std_val = df[target_col].std()
var_val = df[target_col].var()
q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr_val = q3 - q1
skewness_val = df[target_col].skew()

summary_table = pd.DataFrame({
    'Parameter Statistik': [
        'Mean (Rata-rata)', 'Median (Nilai Tengah)', 'Modus (Nilai Terbanyak)',
        'Standar Deviasi', 'Varians', 'Kuartil 1 (Q1)', 'Kuartil 3 (Q3)',
        'Interquartile Range (IQR)', 'Skewness (Kemiringan)'
    ],
    'Nilai Komputasi': [
        f"{mean_val:.2f}", f"{median_val:.2f}", f"{mode_val:.2f}",
        f"{std_val:.2f}", f"{var_val:.2f}", f"{q1:.2f}", f"{q3:.2f}",
        f"{iqr_val:.2f}", f"{skewness_val:.3f}"
    ]
})

print("=== Ringkasan Parameter Statistik Numerik ===")
display(summary_table)


## 🔎 4. Deteksi Outlier dengan Metode Tukey IQR & Z-Score


In [ ]:
# 1. Metode Tukey IQR
lower_bound = q1 - 1.5 * iqr_val
upper_bound = q3 + 1.5 * iqr_val

outliers_iqr = df[(df[target_col] < lower_bound) | (df[target_col] > upper_bound)]

# 2. Metode Z-Score (|Z| > 3)
z_scores = np.abs(stats.zscore(df[target_col]))
outliers_z = df[z_scores > 3]

print(f"Batas Bawah IQR: {lower_bound:.2f} k | Batas Atas IQR: {upper_bound:.2f} k")
print(f"Jumlah Outlier terdeteksi (Metode IQR): {len(outliers_iqr)}")
print(f"Jumlah Outlier terdeteksi (Metode Z-Score |Z|>3): {len(outliers_z)}\n")

print("Daftar Transaksi Outlier (Tukey IQR):")
display(outliers_iqr[['order_id', 'category', 'quantity', 'price_per_unit_k', 'total_amount_k']])


## 📈 5. Visualisasi Distribusi dan Posisi Outlier


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Boxplot dengan penanda outlier
sns.boxplot(x=df[target_col], ax=axes[0], color='skyblue', flierprops={'markerfacecolor':'red', 'markersize':8})
axes[0].set_title(f'Boxplot {target_col} (Deteksi Outlier)', fontweight='bold')
axes[0].set_xlabel('Total Transaksi (dalam Ribu IDR)')

# Subplot 2: Histogram & KDE dengan garis Mean vs. Median
sns.histplot(df[target_col], kde=True, ax=axes[1], color='teal', bins=25)
axes[1].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean ({mean_val:.1f})')
axes[1].axvline(median_val, color='gold', linestyle='-', linewidth=2, label=f'Median ({median_val:.1f})')
axes[1].set_title('Distribusi Nilai Total Transaksi & Skewness', fontweight='bold')
axes[1].set_xlabel('Total Transaksi (dalam Ribu IDR)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis

### Q&A
* **Kapan kita harus menggunakan Median daripada Mean?** Ketika data memiliki *skewness* tinggi atau terdapat data pencilan (*outlier* ekstrem). Median bersifat *robust* (kebal) terhadap pengaruh outlier, sedangkan Mean sangat mudah terdistorsi.
* **Mengapa metode Z-Score dan IQR menghasilkan jumlah outlier berbeda?** Z-Score mengasumsikan data berdistribusi normal, sedangkan metode IQR adalah non-parametrik yang berbasis kuartil nyata dataset.

### Data Analysis Key Findings
* Nilai Mean total transaksi (**Rp {mean_val:.2f}k**) lebih besar dari nilai Median (**Rp {median_val:.2f}k**), menunjukkan distribusi *Right-Skewed* (condong ke kanan) dengan nilai *skewness* positif.
* Ditemukan data transaksi ekstrem (hingga Rp 4.500k) yang teridentifikasi sebagai *outlier* signifikan melalui batas atas Tukey IQR.

### Insights or Next Steps
* Untuk pemodelan prediktif berbasis linier, outlier ekstrem perlu ditransformasi (misal: $\log(x)$) atau dilakukan *winsorization* agar tidak merusak estimasi kuadrat terkecil (OLS).
